# Day G6 — LLM 활용: 임상 텍스트, 문헌 요약, 코드 보조, 그리고 검증

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/G-ai-biomed/day-G6.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/G-ai-biomed/day-G6.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — TF-IDF + 로지스틱 회귀로 낙상 언급 분류

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
# 가상 진료 메모: '낙상 언급 있음(1) / 없음(0)'을 분류. 문장은 템플릿으로 새로 만든 것이다.
pos = ["patient reports a fall at home last week", "fell while turning in the kitchen",
       "two falls since the last visit", "near fall and one fall reported by caregiver",
       "fall with bruise on left hip", "history of recurrent falls, uses a cane now"]
neg = ["no falls reported since last visit", "denies any fall, gait stable",
       "tremor worse in the evening", "sleep improved, no new complaints",
       "walking 30 minutes daily without problems", "medication timing adjusted, no falls"]
openers = ["", "follow-up: ", "today ", "per spouse, ", "clinic note: "]
X, y = [], []
for _ in range(60):
    lab = int(rng.integers(2))
    X.append(rng.choice(openers) + rng.choice(pos if lab else neg)); y.append(lab)

pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print("교차검증 정확도:", np.round(cross_val_score(pipe, X, y, cv=cv), 2))

pipe.fit(X, y)
tests = ["no fall today", "denies falls but reports near fall", "patient fell twice"]
for t in tests:
    print(f"{t!r:<40} → P(낙상 언급) = {pipe.predict_proba([t])[0, 1]:.2f}")
w = dict(zip(pipe[0].get_feature_names_out(), pipe[1].coef_[0]))
print("가중치가 큰 단어:", sorted(w, key=w.get, reverse=True)[:5])
print("가중치가 작은(음수) 단어:", sorted(w, key=w.get)[:5])

### 예시 2 — LLM 요약을 원자료와 대조하는 점검기

In [ ]:
import re

# 원자료(내가 가진 표)와, LLM이 썼다고 가정한 요약 문단을 대조한다. 둘 다 연습용 가상 텍스트다.
source = {"n_participants": 48, "n_pd": 24, "mean_age": 67.3, "auc": 0.71, "auc_ci": (0.58, 0.83)}
summary = ("We analysed 48 participants (24 with PD). Mean age was 67.3 years. "
           "The model reached an AUC of 0.81 (95% CI 0.58–0.83), "
           "confirming that gait features diagnose Parkinson's disease, "
           "consistent with Smith et al. (2019).")

allowed = {str(v) for v in source.values() if not isinstance(v, tuple)}
allowed |= {str(x) for x in source["auc_ci"]}
for sent in re.split(r"(?<!et al\.)(?<=\.)\s+", summary):
    nums = re.findall(r"\d+\.\d+|\d+", sent.replace("95%", ""))
    bad = [n for n in nums if n not in allowed and not re.fullmatch(r"(19|20)\d\d", n)]
    flags = []
    if bad:
        flags.append(f"원자료에 없는 숫자 {bad}")
    if re.search(r"\b(diagnos\w*|confirm\w*|prove\w*|cure\w*)\b", sent, re.I):
        flags.append("과장 표현(진단·확증)")
    if re.search(r"et al\.|\(\d{4}\)", sent):
        flags.append("인용: 원문 실재·내용 직접 확인 필요")
    print(("[확인 필요] " if flags else "[통과]     ") + sent + (("\n            ↳ " + "; ".join(flags)) if flags else ""))

### 예시 3 — 실행 확인 안 됨(네트워크·API 키 필요): LLM API 호출의 형태

LLM 서비스는 보통 HTTP API로 부릅니다. 형태만 보입니다. 서비스마다 함수 이름이 다르고, API 키와 기관의 데이터 사용 승인이 필요하며, 이 환경에서는 실행하지 않았습니다(**실행 확인 안 됨**).

In [ ]:
# 실행 확인 안 됨: 외부 API, 키 필요. 환자 정보를 넣지 말 것.
prompt = ("아래 가상 진료 메모에 낙상 언급이 있으면 1, 없으면 0만 답하라.\n"
          "메모: denies falls but reports near fall")
# answer = client.generate(model="<모델 이름>", prompt=prompt, temperature=0)
# 결과를 사람이 라벨한 평가셋과 비교해 정확도·민감도를 계산한다(Day 34).